# Member 4: Evaluation, ROC Analysis & Final Report
## ECON 7970 Applied Predictive Modeling — Team 10
**Author: Joey (Member 4)**

## 0. Setup

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, confusion_matrix,
                             accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
pd.set_option('display.float_format', '{:.4f}'.format)
print("Libraries loaded")

## 1. Load Data from Member 1

In [ ]:
X_test_full = pd.read_csv("data/processed/test_features.csv")
y_test_raw  = pd.read_csv("data/processed/test_target.csv")
le = LabelEncoder()
y_test = le.fit_transform(y_test_raw['y'])
print(f"Test shape: {X_test_full.shape}, Positive rate: {y_test.mean()*100:.1f}%")

## 2. Train Models on All 4 Scenarios

In [ ]:
cat_cols = ['job','marital','education','default','housing','loan','contact','poutcome']

def load_scenario(sfile, drop_extra):
    tr = pd.read_csv(f"data/processed/{sfile}")
    y_tr = le.transform(tr['y'])
    drop_tr = [c for c in ['y','y_num'] + drop_extra if c in tr.columns]
    drop_te = [c for c in ['y_num'] + drop_extra if c in X_test_full.columns]
    tr2 = tr.drop(columns=drop_tr)
    te2 = X_test_full.drop(columns=drop_te)
    tr_enc = pd.get_dummies(tr2, columns=[c for c in cat_cols if c in tr2.columns])
    te_enc = pd.get_dummies(te2, columns=[c for c in cat_cols if c in te2.columns])
    tr_enc, te_enc = tr_enc.align(te_enc, join='left', axis=1, fill_value=0)
    te_enc = te_enc.reindex(columns=tr_enc.columns, fill_value=0)
    return tr_enc.values.astype(float), y_tr, te_enc.values.astype(float)

S_CONFIGS = {
    'S1': ('train_s1.csv', []),
    'S2': ('train_s2.csv', ['duration']),
    'S3': ('train_s3.csv', ['duration','campaign','pdays','previous','poutcome',
                             'contact','day','month_aug','month_dec','month_feb',
                             'month_jan','month_jul','month_jun','month_mar',
                             'month_may','month_nov','month_oct','month_sep']),
    'S4': ('train_s4.csv', ['age','job','marital','education','default','balance',
                             'housing','loan','duration','campaign','contact','day',
                             'isretired','isstudent','balance_per_call',
                             'month_aug','month_dec','month_feb','month_jan',
                             'month_jul','month_jun','month_mar','month_may',
                             'month_nov','month_oct','month_sep']),
}
SCENARIO_LABELS = {
    'S1':'S1: All Features (with duration)','S2':'S2: Realistic (no duration)',
    'S3':'S3: Demographics Only','S4':'S4: Previous Campaign Only',
}
FAST_MODELS = {
    'KNN':           KNeighborsClassifier(n_neighbors=15),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost':       xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6,
                                        use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1),
    'Logistic Reg':  LogisticRegression(max_iter=300, random_state=42, n_jobs=-1),
}
COLORS = {'KNN':'#2196F3','Random Forest':'#4CAF50','XGBoost':'#E91E63','Logistic Reg':'#FF9800'}

print("Training all models (1-2 min)...")
results_store = {}
for s_name, (sfile, drop_cols) in S_CONFIGS.items():
    X_tr, y_tr, X_te = load_scenario(sfile, drop_cols)
    sc = StandardScaler(); X_tr_sc = sc.fit_transform(X_tr); X_te_sc = sc.transform(X_te)
    sm = SMOTE(random_state=42); X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_sc, y_tr)
    for m_name, model in FAST_MODELS.items():
        for use_smote in [False, True]:
            Xf = X_tr_sm if use_smote else X_tr_sc; yf = y_tr_sm if use_smote else y_tr
            mdl = type(model)(**model.get_params()); mdl.fit(Xf, yf)
            results_store[(m_name, s_name, use_smote)] = (mdl.predict_proba(X_te_sc)[:,1], mdl.predict(X_te_sc))
    print(f"  {s_name} done.")
print("Done!")

## 3. ROC Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('ROC Curves - All Models, All Scenarios (Without SMOTE)', fontsize=14, fontweight='bold')
for ax, s_name in zip(axes.flatten(), S_CONFIGS):
    for m_name in FAST_MODELS:
        proba, _ = results_store[(m_name, s_name, False)]
        fpr, tpr, _ = roc_curve(y_test, proba)
        ax.plot(fpr, tpr, color=COLORS[m_name], lw=2, label=f'{m_name} (AUC={auc(fpr,tpr):.3f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5); ax.set_title(SCENARIO_LABELS[s_name], fontsize=10)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(loc='lower right',fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("results/figures/roc_curves.png",dpi=150,bbox_inches='tight'); plt.show()

## 4. Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('PR Curves - All Models (With SMOTE)', fontsize=14, fontweight='bold')
for ax, s_name in zip(axes.flatten(), S_CONFIGS):
    for m_name in FAST_MODELS:
        proba, _ = results_store[(m_name, s_name, True)]
        prec, rec, _ = precision_recall_curve(y_test, proba)
        ax.plot(rec, prec, color=COLORS[m_name], lw=2, label=f'{m_name} AP={average_precision_score(y_test,proba):.3f}')
    base=y_test.sum()/len(y_test); ax.axhline(base,color='gray',linestyle='--',lw=1.2,label=f'Base {base:.2f}')
    ax.set_title(SCENARIO_LABELS[s_name],fontsize=10); ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(loc='upper right',fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("results/figures/pr_curves.png",dpi=150,bbox_inches='tight'); plt.show()

## 5. Confusion Matrices - Top 3 Models (S2)

In [ ]:
s2_f1 = {m: f1_score(y_test, results_store[(m,'S2',True)][1]) for m in FAST_MODELS}
top3  = sorted(s2_f1, key=s2_f1.get, reverse=True)[:3]
print("Top 3:", top3)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Confusion Matrices - Top 3 Models, S2 Realistic (With SMOTE)', fontsize=13, fontweight='bold')
for ax, m_name in zip(axes, top3):
    _, pred = results_store[(m_name,'S2',True)]
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No','Yes'], yticklabels=['No','Yes'], annot_kws={'size':14})
    p=precision_score(y_test,pred); r=recall_score(y_test,pred); f=f1_score(y_test,pred)
    ax.set_title(f'{m_name}\nP={p:.3f} R={r:.3f} F1={f:.3f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.savefig("results/figures/confusion_matrices/cm_top3_smote.png",dpi=150,bbox_inches='tight'); plt.show()

## 6. Duration Inflation

In [ ]:
rows=[]; mlist=list(FAST_MODELS.keys())
for m in mlist:
    for sf in [False,True]:
        a1=roc_auc_score(y_test,results_store[(m,'S1',sf)][0]); a2=roc_auc_score(y_test,results_store[(m,'S2',sf)][0])
        rows.append({'Model':m,'SMOTE':'Yes' if sf else 'No','AUC_S1':round(a1,4),'AUC_S2':round(a2,4),'Inflation_%':round((a1-a2)/a2*100,2)})
df_infl=pd.DataFrame(rows); print(df_infl.to_string(index=False))
fig,ax=plt.subplots(figsize=(10,6)); x=np.arange(len(mlist)); w=0.35
v_no=[df_infl[(df_infl.Model==m)&(df_infl.SMOTE=='No')]['Inflation_%'].values[0] for m in mlist]
v_yes=[df_infl[(df_infl.Model==m)&(df_infl.SMOTE=='Yes')]['Inflation_%'].values[0] for m in mlist]
ax.bar(x-w/2,v_no,w,label='No SMOTE',color='#2196F3',alpha=0.85); ax.bar(x+w/2,v_yes,w,label='With SMOTE',color='#FF9800',alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(mlist,fontsize=11); ax.set_ylabel('Duration Inflation (%)'); ax.set_title('Duration Inflation (S1 vs S2)',fontweight='bold')
ax.legend(); ax.axhline(0,color='k',lw=0.8); ax.grid(axis='y',alpha=0.3); plt.tight_layout(); plt.show()

## 7. Scenario Comparison (S2 vs S3 vs S4)

In [ ]:
sc_rows=[]
for m in FAST_MODELS:
    for s in ['S2','S3','S4']:
        proba,pred=results_store[(m,s,True)]
        sc_rows.append({'Model':m,'Scenario':s,'ROC-AUC':round(roc_auc_score(y_test,proba),4),'Recall':round(recall_score(y_test,pred),4)})
df_sc=pd.DataFrame(sc_rows)
fig,axes=plt.subplots(1,2,figsize=(16,6))
for ax,metric in zip(axes,['ROC-AUC','Recall']):
    piv=df_sc.pivot(index='Model',columns='Scenario',values=metric)[['S2','S3','S4']]
    xp=np.arange(len(piv)); ww=0.25
    for i,(col,color) in enumerate(zip(['S2','S3','S4'],['#2196F3','#4CAF50','#FF9800'])):
        ax.bar(xp+i*ww,piv[col],ww,label=col,color=color,alpha=0.85)
    ax.set_xticks(xp+ww); ax.set_xticklabels(piv.index); ax.set_title(f'{metric} by Scenario',fontweight='bold'); ax.legend(); ax.grid(axis='y',alpha=0.3)
plt.suptitle('Scenario Comparison (With SMOTE)',fontweight='bold'); plt.tight_layout(); plt.show()

## 8. SMOTE Impact on Recall

In [ ]:
fig,ax=plt.subplots(figsize=(12,6)); mlist2=list(FAST_MODELS.keys()); xp=np.arange(len(mlist2)); ww=0.2
for si,s_name in enumerate(['S1','S2']):
    r_no=[recall_score(y_test,results_store[(m,s_name,False)][1]) for m in mlist2]
    r_yes=[recall_score(y_test,results_store[(m,s_name,True)][1]) for m in mlist2]
    pal=['#2196F3','#4CAF50']
    ax.bar(xp+(si*2-1.5)*ww,r_no,ww,label=f'{s_name} No SMOTE',color=pal[si],alpha=0.6)
    ax.bar(xp+(si*2-0.5)*ww,r_yes,ww,label=f'{s_name} With SMOTE',color=pal[si],alpha=1.0,hatch='//')
ax.set_xticks(xp); ax.set_xticklabels(mlist2,fontsize=11); ax.set_ylabel('Recall')
ax.set_title('SMOTE Impact on Recall',fontweight='bold'); ax.legend(); ax.grid(axis='y',alpha=0.3); ax.set_ylim(0,1.1)
plt.tight_layout(); plt.show()

## 9. Feature Importance (RF + XGBoost, S2)

In [ ]:
tr_s2=pd.read_csv("data/processed/train_s2.csv")
drop_s2=[c for c in ['duration','y','y_num'] if c in tr_s2.columns]
tr_enc2=pd.get_dummies(tr_s2.drop(columns=drop_s2),columns=[c for c in cat_cols if c in tr_s2.columns])
feat_names=tr_enc2.columns.tolist(); X_tr2=tr_enc2.values.astype(float); y_tr2=le.transform(tr_s2['y'])
sc2=StandardScaler(); X_tr2_sc=sc2.fit_transform(X_tr2)
sm2=SMOTE(random_state=42); X_sm2,y_sm2=sm2.fit_resample(X_tr2_sc,y_tr2)
rf2=RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42,n_jobs=-1); rf2.fit(X_sm2,y_sm2)
xgb2=xgb.XGBClassifier(n_estimators=100,use_label_encoder=False,eval_metric='logloss',random_state=42,n_jobs=-1); xgb2.fit(X_sm2,y_sm2)
comb=((pd.Series(rf2.feature_importances_,index=feat_names)+pd.Series(xgb2.feature_importances_,index=feat_names))/2).sort_values(ascending=False).head(15)
fig,ax=plt.subplots(figsize=(10,7)); colors=['#E91E63' if i<3 else '#2196F3' for i in range(len(comb))]
comb.sort_values().plot.barh(ax=ax,color=colors[::-1],alpha=0.85)
ax.set_xlabel('Avg Importance'); ax.set_title('Top 15 Features (RF + XGBoost, S2)',fontweight='bold'); ax.grid(axis='x',alpha=0.3)
plt.tight_layout(); plt.show(); print("Top 3:", comb.index[:3].tolist())

## 10. Threshold Optimization

In [ ]:
best_m=max(FAST_MODELS.keys(),key=lambda m: roc_auc_score(y_test,results_store[(m,'S2',True)][0]))
best_proba_sm=results_store[(best_m,'S2',True)][0]; print(f"Best model: {best_m}")
prec_arr,rec_arr,thr_arr=precision_recall_curve(y_test,best_proba_sm)
f1_arr=np.where((prec_arr+rec_arr)==0,0,2*prec_arr*rec_arr/(prec_arr+rec_arr))
best_idx=np.argmax(f1_arr); best_thr=thr_arr[best_idx]
fig,ax=plt.subplots(figsize=(10,6))
ax.plot(thr_arr,f1_arr[:-1],label='F1',color='#E91E63',lw=2); ax.plot(thr_arr,prec_arr[:-1],label='Precision',color='#2196F3',lw=2); ax.plot(thr_arr,rec_arr[:-1],label='Recall',color='#4CAF50',lw=2)
ax.axvline(best_thr,color='gray',linestyle='--',lw=1.5,label=f'Optimal={best_thr:.3f}'); ax.scatter([best_thr],[f1_arr[best_idx]],color='#E91E63',s=100,zorder=5)
ax.set_xlabel('Threshold'); ax.set_ylabel('Score'); ax.set_title(f'Threshold Optimization - {best_m} + SMOTE (S2)',fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Optimal: {best_thr:.3f} | P={prec_arr[best_idx]:.3f} R={rec_arr[best_idx]:.3f} F1={f1_arr[best_idx]:.3f}")

## 11. Business Recommendation

In [ ]:
df_biz=pd.DataFrame({'prob':best_proba_sm,'actual':y_test})
df_biz=df_biz.sort_values('prob',ascending=False).reset_index(drop=True)
total_buyers=df_biz['actual'].sum()
cum_pct_called=(np.arange(1,len(df_biz)+1))/len(df_biz)*100
cum_pct_buyers=df_biz['actual'].cumsum()/total_buyers*100
fig,ax=plt.subplots(figsize=(10,6))
ax.plot(cum_pct_called,cum_pct_buyers,color='#E91E63',lw=2.5,label='Model Lift Curve'); ax.plot([0,100],[0,100],'k--',lw=1.5,alpha=0.5,label='Random Baseline')
for pct in [10,20,30]:
    idx=int(pct/100*len(df_biz))-1; bc=cum_pct_buyers.iloc[idx]
    ax.scatter([pct],[bc],s=80,zorder=5,color='#E91E63')
    ax.annotate(f'Top {pct}% -> {bc:.0f}% buyers',xy=(pct,bc),xytext=(pct+4,bc-10),fontsize=9,arrowprops=dict(arrowstyle='->',color='gray'))
ax.set_xlabel('% Customers Called'); ax.set_ylabel('% Buyers Reached'); ax.set_title('Cumulative Gains Curve',fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
biz_rows=[]; 
for pct in [10,20,30,40,50]:
    idx=int(pct/100*len(df_biz))-1; bc=cum_pct_buyers.iloc[idx]
    biz_rows.append({'Top_%':f'{pct}%','Buyers_Caught':f'{bc:.1f}%','Calls_Saved':f'{100-pct}%','Lift':f'{bc/pct:.2f}x'})
print(pd.DataFrame(biz_rows).to_string(index=False))
print("RECOMMENDATION: Call top 20% -> catch 75.8% of buyers, save 80% of calls!")

## 12. Final Comparison Table

In [ ]:
df_final=pd.read_csv("results/tables/final_comparison.csv")
best=df_final[df_final.SMOTE=='Yes'].sort_values('ROC-AUC',ascending=False).groupby('Scenario').first()
print("Best model per scenario (with SMOTE):")
print(best[['Model','Accuracy','Recall','F1','ROC-AUC']].to_string())

## 13. Interpretability Trade-off

In [ ]:
df_int=pd.read_csv("results/tables/interpretability_tradeoff.csv")
cols=['Model','Interpretable','ROC-AUC_NoSMOTE','ROC-AUC_SMOTE','Recall_SMOTE']
print(df_int[cols].to_string(index=False))
msg = ("Conclusion: XGBoost is 13.5 AUC pts better (0.902 vs 0.767). "
       "LR has higher recall with SMOTE. Use XGBoost for targeting, LR for explanation.")
print(msg)

## 14. Summary

| Question | Answer |
|----------|--------|
| Best model | XGBoost + SMOTE (S2) |
| ROC-AUC | 0.902 |
| Optimal threshold | 0.44 |
| Call top 20% | Catch 75.8% buyers, save 80% calls |
| Top 3 features | housing_yes, poutcome_success, campaign |
| Duration inflation (XGBoost) | 1.7% |

*Member 4 (Joey) - ECON 7970 Team 10 - April 2026*